In [1]:

import os
import random
import shutil
import json
import datetime
import pathlib

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import h5py

from PIL import Image
from tqdm import tqdm
from scipy.ndimage import (          # FIX: direct import, not via morphology submodule
    gaussian_filter,
    map_coordinates,
    binary_erosion,                  # was: morphology.binary_erosion
    binary_dilation,                 # was: morphology.binary_dilation
)

import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

# ---- Device ----
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Device: {device}")

NUM_BINS = 8   # 8 bins × 22.5° covers [0, 180)



Device: mps


In [2]:

DATA_ROOT    = "/Users/althafali/Downloads/ARTIFEX/implementation/datasets/VanGoghPaintingsData/VincentVanGogh"
WORK_ROOT    = "/Users/althafali/Downloads/ARTIFEX/implementation"

PROCESSED    = os.path.join(WORK_ROOT, "data/processed")
H5_PATH      = os.path.join(WORK_ROOT, "brushstroke_features.h5")
MANIFEST_PATH = os.path.join(PROCESSED, "split_manifest.json")


print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"PROCESSED  : {PROCESSED}")
print(f"H5_PATH    : {H5_PATH}")

# Create base folders
for split in ['train', 'val', 'test']:
    for sub in ['original', 'masked', 'masks',
                'mask_pool',               
                'augmented_masks',
                'augmented_holes']:
        os.makedirs(os.path.join(PROCESSED, split, sub), exist_ok=True)

print("Folder structure created.")



DATA_ROOT  : /Users/althafali/Downloads/ARTIFEX/implementation/datasets/VanGoghPaintingsData/VincentVanGogh
PROCESSED  : /Users/althafali/Downloads/ARTIFEX/implementation/data/processed
H5_PATH    : /Users/althafali/Downloads/ARTIFEX/implementation/brushstroke_features.h5
Folder structure created.


In [3]:

def find_all_images(root_path):
    exts = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}
    found = []
    for r, _, files in os.walk(root_path):
        for f in files:
            if any(f.endswith(e) for e in exts):
                found.append(os.path.join(r, f))
    return found

all_image_paths = find_all_images(DATA_ROOT)
print(f"Total images found: {len(all_image_paths)}")

# Validate
valid_images = []
for p in tqdm(all_image_paths, desc="Validating"):
    try:
        img = Image.open(p)
        img.verify()
        valid_images.append(p)
    except Exception:
        print(f"  Skipping corrupt: {os.path.basename(p)}")

print(f"Valid images: {len(valid_images)}/{len(all_image_paths)}")
all_image_paths = valid_images

Total images found: 2025


Validating: 100%|██████████| 2025/2025 [00:00<00:00, 5535.38it/s]

Valid images: 2025/2025


In [4]:

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

all_image_paths_sorted = sorted(all_image_paths)   # deterministic start
random.shuffle(all_image_paths_sorted)

total = len(all_image_paths_sorted)
train_end = int(total * 0.70)
val_end   = train_end + int(total * 0.15)

train_paths = all_image_paths_sorted[:train_end]
val_paths   = all_image_paths_sorted[train_end:val_end]
test_paths  = all_image_paths_sorted[val_end:]

print(f"Train: {len(train_paths)}  Val: {len(val_paths)}  Test: {len(test_paths)}")

# Save manifest — permanent record of source → processed filename mapping
manifest = {}
for split, paths in [('train', train_paths), ('val', val_paths), ('test', test_paths)]:
    prefix = f"vangogh_{split}"
    manifest[split] = [
        {
            'source_path': src,
            'processed_filename': f"{prefix}_{idx:04d}.png",
            'split': split
        }
        for idx, src in enumerate(paths)
    ]

os.makedirs(PROCESSED, exist_ok=True)
with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"Manifest saved: {MANIFEST_PATH}")



Train: 1417  Val: 303  Test: 305
Manifest saved: /Users/althafali/Downloads/ARTIFEX/implementation/data/processed/split_manifest.json


In [5]:

def preprocess_image(src_path, dst_path, size=(512, 512)):
    try:
        img = Image.open(src_path).convert('RGB')
        img = img.resize(size, Image.LANCZOS)
        img.save(dst_path, 'PNG')
        return True
    except Exception as e:
        print(f"  Error: {src_path}: {e}")
        return False

for split, paths in [('train', train_paths), ('val', val_paths), ('test', test_paths)]:
    prefix  = f"vangogh_{split}"
    out_dir = os.path.join(PROCESSED, split, 'original')
    ok = 0
    for idx, src in enumerate(tqdm(paths, desc=f"Preprocessing {split}")):
        dst = os.path.join(out_dir, f"{prefix}_{idx:04d}.png")
        if preprocess_image(src, dst):
            ok += 1
    print(f"  {split}: {ok}/{len(paths)} processed")

# Verify
print("\nVerification:")
for split in ['train', 'val', 'test']:
    d = os.path.join(PROCESSED, split, 'original')
    n = len([f for f in os.listdir(d) if f.endswith('.png')])
    expected = {'train': len(train_paths), 'val': len(val_paths), 'test': len(test_paths)}[split]
    status = "OK" if n == expected else f"MISMATCH (got {n}, expected {expected})"
    print(f"  {split}: {n} images  {status}")



Preprocessing train: 100%|██████████| 1417/1417 [00:55<00:00, 25.53it/s]


  train: 1417/1417 processed


Preprocessing val: 100%|██████████| 303/303 [00:12<00:00, 25.08it/s]


  val: 303/303 processed


Preprocessing test: 100%|██████████| 305/305 [00:11<00:00, 26.22it/s]

  test: 305/305 processed

Verification:
  train: 1417 images  OK
  val: 303 images  OK
  test: 305 images  OK


In [6]:

ORIGINAL_MASKS = "/Users/althafali/Downloads/ARTIFEX/implementation/datasets/mask"

all_base_masks = [f for f in os.listdir(ORIGINAL_MASKS) if f.endswith('.png')]
print(f"Base masks found: {len(all_base_masks)}")

random.seed(SEED)   # re-seed so mask split is always identical
random.shuffle(all_base_masks)

m_train_end = int(len(all_base_masks) * 0.70)
m_val_end   = m_train_end + int(len(all_base_masks) * 0.15)

split_masks = {
    'train': all_base_masks[:m_train_end],
    'val':   all_base_masks[m_train_end:m_val_end],
    'test':  all_base_masks[m_val_end:],
}

for split, masks in split_masks.items():
    dst_dir = os.path.join(PROCESSED, split, 'mask_pool')   # FIX: mask_pool not masks
    for f in masks:
        shutil.copy(os.path.join(ORIGINAL_MASKS, f), os.path.join(dst_dir, f))
    print(f"  {split}: {len(masks)} base masks copied to mask_pool/")


Base masks found: 49
  train: 34 base masks copied to mask_pool/
  val: 7 base masks copied to mask_pool/
  test: 8 base masks copied to mask_pool/


In [7]:
# ============================================================================
# 2. HELPER FUNCTIONS (updated)
# ============================================================================
def load_mask(path, size=(512, 512)):
    mask = Image.open(path).convert('L')
    mask = mask.resize(size, Image.NEAREST)
    arr = np.array(mask)
    return (arr > 128).astype(np.uint8) * 255

def save_mask(mask, path):
    Image.fromarray(mask.astype(np.uint8)).save(path)

def mask_coverage_pct(mask):
    return float(np.mean(mask > 0) * 100.0)

def rotate_mask(img, angle):
    """Exact rotation for 90/180/270 to avoid warp artifacts."""
    angle = angle % 360
    if angle in (0, 90, 180, 270):
        return np.rot90(img, angle // 90).copy()
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_NEAREST)

def scale_mask(img, factor):
    h, w = img.shape
    nw, nh = int(w * factor), int(h * factor)
    scaled = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_NEAREST)

    if factor < 1.0:
        out = np.zeros((h, w), dtype=np.uint8)
        py, px = (h - nh) // 2, (w - nw) // 2
        out[py:py+nh, px:px+nw] = scaled
        return out
    else:
        sy, sx = (nh - h) // 2, (nw - w) // 2
        return scaled[sy:sy+h, sx:sx+w]

def elastic_deform(img, alpha=8, sigma=4, seed=None):
    """Deterministic if seed is provided."""
    rng = np.random.RandomState(seed)
    shape = img.shape
    dx = gaussian_filter((rng.rand(*shape) * 2 - 1), sigma) * alpha
    dy = gaussian_filter((rng.rand(*shape) * 2 - 1), sigma) * alpha
    x, y = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
    idxs = (y + dy).ravel(), (x + dx).ravel()
    out = map_coordinates(img, idxs, order=1, mode='reflect').reshape(shape)
    return (out > 128).astype(np.uint8) * 255

def adjust_crack_coverage(mask, target, tol=0.02, max_iter=60):
    """
    For crack masks only (low coverage: 5/10/15).
    Mild organic dilation/erosion.
    """
    adj = mask.copy()
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    for _ in range(max_iter):
        current = np.mean(adj > 0)
        if abs(current - target) <= tol:
            break
        if current < target:
            adj = cv2.dilate(adj, kernel, iterations=1)
        else:
            adj = cv2.erode(adj, kernel, iterations=1)
    return (adj > 0).astype(np.uint8) * 255

def augment_mask(mask, seed_base=0):
    return [
        mask,
        np.fliplr(mask).copy(),
        np.flipud(mask).copy(),
        rotate_mask(mask, 90),
        rotate_mask(mask, 180),
        rotate_mask(mask, 270),
        scale_mask(mask, 0.8),
        scale_mask(mask, 1.2),
        elastic_deform(mask, alpha=8, sigma=4, seed=seed_base + 1000),
    ]

def load_mask_cache(folder, suffix_filter=None):
    files = [f for f in os.listdir(folder) if f.endswith('.png')]
    if suffix_filter:
        files = [f for f in files if suffix_filter in f]
    cache = {}
    for fname in tqdm(files, desc=f"  Caching {os.path.basename(folder)}", leave=False):
        cache[fname] = load_mask(os.path.join(folder, fname))
    return cache

def shift_mask(mask, dx, dy):
    h, w = mask.shape
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    shifted = cv2.warpAffine(
        mask, M, (w, h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )
    return shifted

In [8]:
# ============================================================================
# 3. CRACK MASK AUGMENTATION (FIXED: low coverage only 05/10/15)
# ============================================================================
def augment_crack_masks(split):
    """
    Real crack masks + augmentation -> only low coverage buckets (5/10/15).
    """
    src_dir = os.path.join(PROCESSED, split, 'mask_pool')
    dst_dir = os.path.join(PROCESSED, split, 'augmented_masks')
    os.makedirs(dst_dir, exist_ok=True)

    # Optional cleanup
    for f in os.listdir(dst_dir):
        if f.endswith('.png'):
            os.remove(os.path.join(dst_dir, f))

    masks = sorted([f for f in os.listdir(src_dir) if f.endswith('.png')])
    idx = 0

    for m_i, mask_file in enumerate(tqdm(masks, desc=f"  Augmenting crack masks [{split}]")):
        base_mask = load_mask(os.path.join(src_dir, mask_file))
        aug_list = augment_mask(base_mask, seed_base=SEED + m_i * 100)

        for aug_idx, aug in enumerate(aug_list):
            for cov_pct in [5, 10, 15]:
                target = cov_pct / 100.0
                adj = adjust_crack_coverage(aug, target, tol=0.02, max_iter=60)

                actual = mask_coverage_pct(adj)
                if abs(actual - cov_pct) > 3.0:
                    print(f"    Warning: {mask_file} aug{aug_idx} cov{cov_pct:02d} actual={actual:.2f}")

                stem = os.path.splitext(mask_file)[0]
                out_name = f"{stem}_aug{aug_idx}_cov{cov_pct:02d}.png"
                save_mask(adj, os.path.join(dst_dir, out_name))
                idx += 1

    print(f"  {split}: generated {idx} augmented crack masks")

for split in ['train', 'val', 'test']:
    augment_crack_masks(split)


  Augmenting crack masks [train]: 100%|██████████| 34/34 [00:03<00:00,  9.96it/s]


  train: generated 918 augmented crack masks


  Augmenting crack masks [val]: 100%|██████████| 7/7 [00:00<00:00, 10.78it/s]


  val: generated 189 augmented crack masks


  Augmenting crack masks [test]: 100%|██████████| 8/8 [00:00<00:00, 11.23it/s]

  test: generated 216 augmented crack masks


In [9]:
# ============================================================================
# 4. COMPACT IRREGULAR FLAKE-LIKE HOLE MASK GENERATION (REPLACES old v3 holes)
# ============================================================================
def _largest_components(mask, min_area=200):
    m = (mask > 0).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    comps = []
    for i in range(1, n):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area >= min_area:
            x = int(stats[i, cv2.CC_STAT_LEFT])
            y = int(stats[i, cv2.CC_STAT_TOP])
            w = int(stats[i, cv2.CC_STAT_WIDTH])
            h = int(stats[i, cv2.CC_STAT_HEIGHT])
            comps.append({"label": i, "area": area, "bbox": (x, y, w, h)})
    comps.sort(key=lambda c: c["area"], reverse=True)
    return comps, labels

def _flake_polygon(rng, H, W, center=None, base_radius=None, n_pts_range=(18, 34)):
    if center is None:
        cx = int(rng.integers(W // 4, 3 * W // 4))
        cy = int(rng.integers(H // 4, 3 * H // 4))
    else:
        cx, cy = center

    if base_radius is None:
        base_radius = int(rng.integers(55, 140))

    n_pts = int(rng.integers(n_pts_range[0], n_pts_range[1] + 1))
    angles = np.sort(rng.uniform(0, 2 * np.pi, n_pts))

    pts = []
    phase = rng.uniform(-1.0, 1.0)
    for a in angles:
        r = base_radius * rng.uniform(0.65, 1.25)
        r *= (1.0 + 0.10 * np.sin(3 * a + phase))
        x = int(np.clip(cx + r * np.cos(a), 0, W - 1))
        y = int(np.clip(cy + r * np.sin(a), 0, H - 1))
        pts.append([x, y])

    return np.array(pts, dtype=np.int32)

def _draw_flake(mask, rng, center=None, base_radius=None):
    H, W = mask.shape
    poly = _flake_polygon(rng, H, W, center=center, base_radius=base_radius)
    cv2.fillPoly(mask, [poly], 255)

    # Mild organic boundary refinement
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask[:] = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=1)

    # Carve small chips (ragged edges) - mostly near canvas, harmless if outside flake
    for _ in range(int(rng.integers(2, 6))):
        rr = int(rng.integers(8, 22))
        x = int(rng.integers(0, W))
        y = int(rng.integers(0, H))
        cv2.circle(mask, (x, y), rr, 0, -1)

    # Light cleanup
    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask[:] = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k2, iterations=1)
    mask[:] = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k2, iterations=1)

def check_compact_flake_quality(mask, target_coverage=0.30, tol=0.03):
    """
    Accept/reject rules for hole masks (30/45):
    - compact, few components
    - not spaghetti, not full-frame spread
    """
    H, W = mask.shape
    cov = float(np.mean(mask > 0))
    if abs(cov - target_coverage) > tol:
        return False, {"reason": "coverage", "cov": cov}

    comps, _ = _largest_components(mask, min_area=int(0.002 * H * W))
    if len(comps) == 0:
        return False, {"reason": "no_large_components", "cov": cov}

    if len(comps) > 3:
        return False, {"reason": "too_many_components", "cov": cov, "ncomp": len(comps)}

    total_area = sum(c["area"] for c in comps)
    largest_frac = comps[0]["area"] / max(total_area, 1)
    if largest_frac < 0.45:
        return False, {"reason": "too_fragmented", "cov": cov, "largest_frac": largest_frac}

    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return False, {"reason": "empty_after_ops", "cov": cov}
    x_span = (xs.max() - xs.min() + 1) / W
    y_span = (ys.max() - ys.min() + 1) / H
    if x_span > 0.92 and y_span > 0.92:
        return False, {"reason": "full_frame_spread", "cov": cov, "x_span": x_span, "y_span": y_span}

    contours, _ = cv2.findContours((mask > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    peri = sum(cv2.arcLength(c, True) for c in contours)
    area = float(np.sum(mask > 0))
    compactness_proxy = (peri * peri) / (4 * np.pi * area + 1e-6)
    if compactness_proxy > 80:  # branchy/spaghetti masks tend to be much larger
        return False, {"reason": "too_branchy", "cov": cov, "compactness": compactness_proxy}

    return True, {
        "reason": "ok",
        "cov": cov,
        "ncomp": len(comps),
        "largest_frac": largest_frac,
        "x_span": x_span,
        "y_span": y_span,
    }

def generate_compact_flake_hole_mask(size=(512, 512), target_coverage=0.30, tol=0.02, seed=None, max_tries=220):
    """
    Compact irregular flake-like hole generator.
    No rectangles. No ellipses. No web/spaghetti masks.
    """
    rng = np.random.default_rng(seed)
    H, W = size

    best_mask = None
    best_err = 1e9
    best_info = None

    for _ in range(max_tries):
        mask = np.zeros((H, W), dtype=np.uint8)

        if target_coverage <= 0.30:
            n_flakes = int(rng.choice([1, 2], p=[0.70, 0.30]))
        else:
            n_flakes = int(rng.choice([1, 2, 3], p=[0.45, 0.45, 0.10]))

        # Localized anchor so holes stay compact
        anchor_cx = int(rng.integers(W // 3, 2 * W // 3))
        anchor_cy = int(rng.integers(H // 3, 2 * H // 3))

        for _j in range(n_flakes):
            offset_scale = 35 if target_coverage <= 0.30 else 55
            cx = int(np.clip(anchor_cx + rng.normal(0, offset_scale), 40, W - 40))
            cy = int(np.clip(anchor_cy + rng.normal(0, offset_scale), 40, H - 40))
            base_r = int(rng.integers(65, 120)) if target_coverage <= 0.30 else int(rng.integers(85, 150))
            _draw_flake(mask, rng, center=(cx, cy), base_radius=base_r)

        # Gentle merge if flakes are close
        k_merge = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_merge, iterations=1)

        # Small controlled coverage adjustment (not repeated global bloat)
        cov = float(np.mean(mask > 0))
        k_adj = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        if cov < target_coverage - tol:
            for _ in range(8):
                mask = cv2.dilate(mask, k_adj, iterations=1)
                cov = float(np.mean(mask > 0))
                if cov >= target_coverage - tol:
                    break
        elif cov > target_coverage + tol:
            for _ in range(8):
                mask = cv2.erode(mask, k_adj, iterations=1)
                cov = float(np.mean(mask > 0))
                if cov <= target_coverage + tol:
                    break

        # Final cleanup
        k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k3, iterations=1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k3, iterations=1)
        mask = (mask > 0).astype(np.uint8) * 255

        ok, info = check_compact_flake_quality(mask, target_coverage=target_coverage, tol=max(tol, 0.03))
        err = abs(float(np.mean(mask > 0)) - target_coverage)

        if err < best_err:
            best_err = err
            best_mask = mask.copy()
            best_info = info

        if ok:
            return mask, info

    return best_mask, {"reason": "fallback_best", **(best_info or {})}

def generate_holes_masks(split, num_per_target=100):
    """
    v3-compatible replacement:
    generates compact irregular flake-like holes at 30% and 45%.
    """
    dst_dir = os.path.join(PROCESSED, split, 'augmented_holes')
    os.makedirs(dst_dir, exist_ok=True)

    # Optional cleanup
    for f in os.listdir(dst_dir):
        if f.endswith('.png'):
            os.remove(os.path.join(dst_dir, f))

    counts = {'train': num_per_target, 'val': 30, 'test': 30}
    n = counts[split]
    split_seed_offset = {'train': 0, 'val': 100000, 'test': 200000}[split]
    fallbacks = 0
    total = 0

    for cov_pct in [30, 45]:
        target = cov_pct / 100.0
        for i in tqdm(range(n), desc=f"  Compact flake holes {cov_pct}% [{split}]"):
            seed = split_seed_offset + cov_pct * 1000 + i
            mask, info = generate_compact_flake_hole_mask(
                size=(512, 512),
                target_coverage=target,
                tol=0.02,
                seed=seed,
                max_tries=220
            )

            # Light deterministic transforms (preserve compactness)
            rng = np.random.default_rng(seed + 999)
            if rng.random() > 0.5:
                mask = np.fliplr(mask).copy()
            if rng.random() > 0.5:
                mask = np.flipud(mask).copy()
            k = int(rng.integers(0, 4))
            if k:
                mask = np.rot90(mask, k).copy()

            if info.get("reason") == "fallback_best":
                fallbacks += 1

            out_name = f"holes_{i:04d}_cov{cov_pct}.png"
            save_mask(mask, os.path.join(dst_dir, out_name))
            total += 1

    print(f"  {split}: generated {total} compact hole masks (fallbacks used: {fallbacks})")

for split in ['train', 'val', 'test']:
    generate_holes_masks(split)

  Compact flake holes 45% [train]: 100%|██████████| 100/100 [00:11<00:00,  9.00it/s]


  train: generated 200 compact hole masks (fallbacks used: 0)


  Compact flake holes 45% [val]: 100%|██████████| 30/30 [00:03<00:00,  9.88it/s]


  val: generated 60 compact hole masks (fallbacks used: 0)


  Compact flake holes 45% [test]: 100%|██████████| 30/30 [00:03<00:00,  8.79it/s]

  test: generated 60 compact hole masks (fallbacks used: 0)


In [ ]:
# ============================================================================
# 5. QUICK MASK COUNT CHECKS (v3-style)
# ============================================================================
print("\nMask counts after generation:")
for split in ['train', 'val', 'test']:
    crack_dir = os.path.join(PROCESSED, split, 'augmented_masks')
    holes_dir = os.path.join(PROCESSED, split, 'augmented_holes')

    crack_files = [f for f in os.listdir(crack_dir) if f.endswith('.png')]
    holes_files = [f for f in os.listdir(holes_dir) if f.endswith('.png')]

    c05 = sum(1 for f in crack_files if '_cov05' in f)
    c10 = sum(1 for f in crack_files if '_cov10' in f)
    c15 = sum(1 for f in crack_files if '_cov15' in f)
    h30 = sum(1 for f in holes_files if '_cov30' in f)
    h45 = sum(1 for f in holes_files if '_cov45' in f)

    print(f"  {split}: crack cov05={c05} cov10={c10} cov15={c15}  holes cov30={h30} cov45={h45}")

    assert c05 > 0, f"FAIL: no _cov05 masks in {split}"
    assert c10 > 0, f"FAIL: no _cov10 masks in {split}"
    assert c15 > 0, f"FAIL: no _cov15 masks in {split}"
    assert h30 > 0, f"FAIL: no _cov30 holes in {split}"
    assert h45 > 0, f"FAIL: no _cov45 holes in {split}"
print("All mask suffix assertions passed.")

In [ ]:
# ============================================================================
# 6. CORRUPTION PIPELINE (FIXED: post-shift coverage drift)
# ============================================================================
SPLIT_SEEDS = {'train': 42, 'val': 123, 'test': 456}

# exact target distribution across images
COVERAGE_DIST = [
    ('05', 'crack', 0.20),
    ('10', 'crack', 0.20),
    ('15', 'crack', 0.20),
    ('30', 'holes', 0.20),
    ('45', 'holes', 0.20),
]

TARGETS_PCT = [5, 10, 15, 30, 45]

# Tolerance for actual coverage after shift (percentage points)
COVERAGE_TOL_PCT = {
    '05': 4.0,
    '10': 4.0,
    '15': 4.0,
    '30': 4.0,
    '45': 4.0,
}

# Smaller shift for high coverage holes -> less clipping drift
SHIFT_LIMITS = {
    '05': 50,
    '10': 50,
    '15': 45,
    '30': 35,
    '45': 20,
}

def nearest_target_bucket(cov_pct, targets=TARGETS_PCT):
    return min(targets, key=lambda t: abs(cov_pct - t))

def make_exact_assignments(n_images, coverage_dist, seed):
    """
    Largest-remainder exact assignment so split totals match exactly.
    """
    raw = [n_images * frac for (_, _, frac) in coverage_dist]
    base = [int(math.floor(x)) for x in raw]
    remainder = n_images - sum(base)

    frac_parts = [(i, raw[i] - base[i]) for i in range(len(raw))]
    frac_parts.sort(key=lambda x: x[1], reverse=True)

    counts = base[:]
    for i in range(remainder):
        counts[frac_parts[i][0]] += 1

    assignments = []
    for (cov_str, mask_type, _), count in zip(coverage_dist, counts):
        assignments.extend([(cov_str, mask_type)] * count)

    rng = np.random.default_rng(seed)
    rng.shuffle(assignments)
    return assignments

def create_corrupted_images(split):
    original_dir  = os.path.join(PROCESSED, split, 'original')
    masked_dir    = os.path.join(PROCESSED, split, 'masked')
    mask_save_dir = os.path.join(PROCESSED, split, 'masks')  # applied masks only
    crack_dir     = os.path.join(PROCESSED, split, 'augmented_masks')
    holes_dir     = os.path.join(PROCESSED, split, 'augmented_holes')
    meta_path     = os.path.join(PROCESSED, split, 'metadata.json')

    os.makedirs(masked_dir, exist_ok=True)
    os.makedirs(mask_save_dir, exist_ok=True)

    # Optional cleanup of outputs (avoids stale files from previous runs)
    for folder in [masked_dir, mask_save_dir]:
        for f in os.listdir(folder):
            if f.endswith('.png'):
                os.remove(os.path.join(folder, f))

    print(f"  Loading mask caches for {split}...")
    crack_caches = {
        '05': load_mask_cache(crack_dir, '_cov05'),
        '10': load_mask_cache(crack_dir, '_cov10'),
        '15': load_mask_cache(crack_dir, '_cov15'),
    }
    holes_caches = {
        '30': load_mask_cache(holes_dir, '_cov30'),
        '45': load_mask_cache(holes_dir, '_cov45'),
    }

    for cov, cache in {**crack_caches, **holes_caches}.items():
        assert len(cache) > 0, f"FAIL: empty mask cache for cov={cov} in {split}"

    crack_keys = {k: list(v.keys()) for k, v in crack_caches.items()}
    holes_keys = {k: list(v.keys()) for k, v in holes_caches.items()}

    print(f"  Mask cache sizes — crack: 05={len(crack_caches['05'])} 10={len(crack_caches['10'])} 15={len(crack_caches['15'])}  "
          f"holes: 30={len(holes_caches['30'])} 45={len(holes_caches['45'])}")

    image_files = sorted([f for f in os.listdir(original_dir) if f.endswith('.png')])
    n_images = len(image_files)

    assignments = make_exact_assignments(n_images, COVERAGE_DIST, seed=SPLIT_SEEDS[split])
    rng = np.random.default_rng(SPLIT_SEEDS[split] + 9999)

    metadata = []
    fallback_count = 0

    for img_file, (cov_str, mask_type) in tqdm(
        zip(image_files, assignments),
        desc=f"  Corrupting {split}",
        total=n_images
    ):
        target_pct = int(cov_str)
        tol_pct = COVERAGE_TOL_PCT[cov_str]
        shift_lim = SHIFT_LIMITS[cov_str]

        orig_np = np.array(Image.open(os.path.join(original_dir, img_file)).convert('RGB')) / 255.0

        cache = crack_caches[cov_str] if mask_type == 'crack' else holes_caches[cov_str]
        key_list = crack_keys[cov_str] if mask_type == 'crack' else holes_keys[cov_str]

        best = None
        accepted = None

        # Re-sample mask+shift to keep actual coverage near target after clipping
        for _attempt in range(25):
            mask_fname = str(rng.choice(key_list))
            base_mask = cache[mask_fname]

            dx = int(rng.integers(-shift_lim, shift_lim + 1))
            dy = int(rng.integers(-shift_lim, shift_lim + 1))
            shifted = shift_mask(base_mask, dx, dy)

            if np.sum(shifted > 0) == 0:
                continue

            actual_cov = mask_coverage_pct(shifted)
            diff = abs(actual_cov - target_pct)

            cand = {
                "mask_fname": mask_fname,
                "mask": shifted,
                "dx": dx,
                "dy": dy,
                "actual_cov": actual_cov,
                "diff": diff
            }

            if (best is None) or (diff < best["diff"]):
                best = cand

            if diff <= tol_pct:
                accepted = cand
                break

        if accepted is None:
            accepted = best
            fallback_count += 1

        shifted = accepted["mask"]
        mask_bool = shifted > 0

        corrupted = orig_np.copy()
        corrupted[mask_bool] = 0.0   # keep same corruption style as your v3 baseline

        Image.fromarray((corrupted * 255).astype(np.uint8)).save(os.path.join(masked_dir, img_file))
        save_mask(shifted, os.path.join(mask_save_dir, img_file))

        metadata.append({
            'image_name': img_file,
            'mask_file': accepted["mask_fname"],
            'mask_type': mask_type,
            'target_coverage': target_pct,
            'actual_coverage': round(float(accepted["actual_cov"]), 2),
            'dx': int(accepted["dx"]),
            'dy': int(accepted["dy"]),
            'coverage_error': round(float(accepted["diff"]), 2),
            'resample_fallback_used': bool(accepted["diff"] > tol_pct),
        })

    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"  {split}: {n_images} images corrupted, metadata saved. Fallbacks used: {fallback_count}")
    return metadata

all_metadata = {}
for split in ['train', 'val', 'test']:
    all_metadata[split] = create_corrupted_images(split)


In [ ]:
# ============================================================================
# 7. COVERAGE VERIFICATION (FIXED: no overlapping bucket bug)
# ============================================================================
def verify_coverage_distribution(split):
    meta_path = os.path.join(PROCESSED, split, 'metadata.json')
    with open(meta_path, 'r') as f:
        metadata = json.load(f)

    total = len(metadata)
    targets = [5, 10, 15, 30, 45]

    by_assigned = {t: 0 for t in targets}
    by_actual_nearest = {t: 0 for t in targets}
    confusion = {a: {b: 0 for b in targets} for a in targets}
    errors_by_target = {t: [] for t in targets}
    unmatched = 0
    tol = 4.0

    for entry in metadata:
        assigned = int(entry['target_coverage'])
        actual = float(entry['actual_coverage'])

        by_assigned[assigned] += 1

        nearest = nearest_target_bucket(actual, targets=targets)
        confusion[assigned][nearest] += 1
        errors_by_target[assigned].append(actual - assigned)

        if abs(actual - nearest) <= tol:
            by_actual_nearest[nearest] += 1
        else:
            unmatched += 1

    print(f"\n{split.upper()} Coverage Distribution ({total} images total)")
    print(f"  {'Target':>8}  {'Assigned':>10}  {'Nearest Actual':>14}  {'% of total':>12}")
    print("  " + "-" * 54)
    for t in targets:
        print(f"  {t:>6}%  {by_assigned[t]:>10}  {by_actual_nearest[t]:>14}  {by_actual_nearest[t]/total*100:>11.1f}%")

    if unmatched:
        print(f"  {'Unmatched':>8}  {'':>10}  {unmatched:>14}  {unmatched/total*100:>11.1f}%")
        print(f"  WARNING: {unmatched} images are >±{tol}% away from nearest target bucket.")
    else:
        print(f"  All {total} images are within ±{tol}% of a nearest target bucket. OK")

    print("\n  Mean actual coverage error (actual - assigned target):")
    for t in targets:
        errs = np.array(errors_by_target[t], dtype=np.float32)
        if len(errs) == 0:
            continue
        print(f"    {t:>2}%: mean={errs.mean():+.2f}  std={errs.std():.2f}  min={errs.min():+.2f}  max={errs.max():+.2f}")

    type_counts = {}
    for entry in metadata:
        mt = entry.get('mask_type', 'unknown')
        type_counts[mt] = type_counts.get(mt, 0) + 1
    print(f"  Mask types: {type_counts}")

    print("\n  Assigned -> nearest actual bucket (counts):")
    header = "         " + " ".join([f"{t:>6}%" for t in targets])
    print(header)
    for a in targets:
        row = "  {:>4}% ".format(a) + " ".join([f"{confusion[a][b]:>6}" for b in targets])
        print(row)

for split in ['train', 'val', 'test']:
    verify_coverage_distribution(split)


In [ ]:
# ============================================================================
# 8. FINAL COUNT CHECKS (v3-style)
# ============================================================================
print("\nFolder count check:")
for split in ['train', 'val', 'test']:
    orig  = len([f for f in os.listdir(os.path.join(PROCESSED, split, 'original')) if f.endswith('.png')])
    corr  = len([f for f in os.listdir(os.path.join(PROCESSED, split, 'masked'))   if f.endswith('.png')])
    masks = len([f for f in os.listdir(os.path.join(PROCESSED, split, 'masks'))    if f.endswith('.png')])
    pool  = len([f for f in os.listdir(os.path.join(PROCESSED, split, 'mask_pool')) if f.endswith('.png')])

    ok = "OK" if orig == corr == masks else "MISMATCH"
    print(f"  {split}: original={orig} masked={corr} masks={masks} mask_pool={pool}  {ok}")
    assert orig == corr == masks, f"Count mismatch in {split}"

print("All count checks passed.")

In [9]:

def generate_hole_mask(size=(512, 512), target_coverage=0.30, tol=0.02, seed=None):
    rng = np.random.RandomState(seed)
    H, W = size
    mask = np.zeros((H, W), dtype=np.uint8)
    total = H * W
    attempts = 0

    while np.sum(mask > 0) / total < target_coverage - tol and attempts < 150:
        attempts += 1
        shape_type = rng.choice(['rectangle', 'ellipse', 'blob'])

        if shape_type == 'rectangle':
            rh = rng.randint(50, 200)
            rw = rng.randint(50, 200)
            top  = rng.randint(0, H - rh)
            left = rng.randint(0, W - rw)
            mask[top:top+rh, left:left+rw] = 255

        elif shape_type == 'ellipse':
            cx, cy = rng.randint(50, W-50), rng.randint(50, H-50)
            ax, ay = rng.randint(30, 100), rng.randint(30, 100)
            angle  = rng.randint(0, 180)
            cv2.ellipse(mask, (cx, cy), (ax, ay), angle, 0, 360, 255, -1)

        else:  # blob
            n_pts  = rng.randint(5, 10)
            cx, cy = rng.randint(100, W-100), rng.randint(100, H-100)
            radius = rng.randint(40, 120)
            angles = sorted(rng.uniform(0, 2*np.pi, n_pts))
            pts = []
            for a in angles:
                r = radius * rng.uniform(0.5, 1.5)
                pts.append([
                    int(np.clip(cx + r * np.cos(a), 0, W-1)),
                    int(np.clip(cy + r * np.sin(a), 0, H-1))
                ])
            cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], 255)

        # pull back if overshot
        current = np.sum(mask > 0) / total
        if current > target_coverage + tol:
            kernel = np.ones((5, 5), np.uint8)
            mask   = cv2.erode(mask, kernel, iterations=1)

    return mask

def generate_holes_masks(split, num_per_target=100):
    """Generate hole-style masks at 30% and 45% coverage."""
    dst_dir = os.path.join(PROCESSED, split, 'augmented_holes')
    os.makedirs(dst_dir, exist_ok=True)
    # scale count by split size
    counts  = {'train': num_per_target, 'val': 30, 'test': 30}
    n       = counts[split]
    total   = 0

    for cov_pct in [30, 45]:
        target = cov_pct / 100.0
        for i in tqdm(range(n), desc=f"  Holes {cov_pct}% [{split}]"):
            seed = (cov_pct * 10000 + i + {'train': 0, 'val': 50000, 'test': 100000}[split])
            mask = generate_hole_mask(target_coverage=target, seed=seed)
            mask = (mask > 128).astype(np.uint8) * 255
            # random flip/rotate
            rng  = np.random.RandomState(seed + 1)
            if rng.random() > 0.5: mask = np.fliplr(mask)
            if rng.random() > 0.5: mask = np.flipud(mask)
            k = rng.choice([0, 1, 2, 3])
            if k: mask = np.rot90(mask, k)
            out_name = f"holes_{i:04d}_cov{cov_pct}.png"
            save_mask(mask, os.path.join(dst_dir, out_name))
            total += 1

    print(f"  {split}: generated {total} holes masks")

for split in ['train', 'val', 'test']:
    generate_holes_masks(split)

# Quick verification
print("\nMask counts after generation:")
for split in ['train', 'val', 'test']:
    crack_dir = os.path.join(PROCESSED, split, 'augmented_masks')
    holes_dir = os.path.join(PROCESSED, split, 'augmented_holes')
    crack_files = [f for f in os.listdir(crack_dir) if f.endswith('.png')]
    holes_files = [f for f in os.listdir(holes_dir) if f.endswith('.png')] if os.path.exists(holes_dir) else []
    # verify suffix counts
    c05  = sum(1 for f in crack_files if '_cov05' in f)
    c10  = sum(1 for f in crack_files if '_cov10' in f)
    c15  = sum(1 for f in crack_files if '_cov15' in f)
    h30  = sum(1 for f in holes_files if '_cov30' in f)
    h45  = sum(1 for f in holes_files if '_cov45' in f)
    print(f"  {split}: crack cov05={c05} cov10={c10} cov15={c15}  "
          f"holes cov30={h30} cov45={h45}")
    assert c05 > 0, f"FAIL: no _cov05 masks in {split} — check augment_crack_masks"
    assert c10 > 0, f"FAIL: no _cov10 masks in {split}"
    assert h30 > 0, f"FAIL: no _cov30 holes in {split} — check generate_holes_masks"
    assert h45 > 0, f"FAIL: no _cov45 holes in {split}"
print("All mask suffix assertions passed.")



  Holes 45% [train]: 100%|██████████| 100/100 [00:00<00:00, 199.59it/s]


  train: generated 200 holes masks


  Holes 45% [val]: 100%|██████████| 30/30 [00:00<00:00, 134.87it/s]


  val: generated 60 holes masks


  Holes 45% [test]: 100%|██████████| 30/30 [00:00<00:00, 170.21it/s]

  test: generated 60 holes masks

Mask counts after generation:
  train: crack cov05=306 cov10=306 cov15=306  holes cov30=100 cov45=100
  val: crack cov05=63 cov10=63 cov15=63  holes cov30=30 cov45=30
  test: crack cov05=72 cov10=72 cov15=72  holes cov30=30 cov45=30
All mask suffix assertions passed.


In [10]:
SPLIT_SEEDS = {'train': 42, 'val': 123, 'test': 456}

# Coverage distribution per split
# Each image gets exactly one coverage level chosen from this weighted distribution
COVERAGE_DIST = [
    ('05', 'crack', 0.20),   # 20% of images → 5% crack damage
    ('10', 'crack', 0.20),   # 20%           → 10% crack damage
    ('15', 'crack', 0.20),   # 20%           → 15% crack damage
    ('30', 'holes', 0.20),   # 20%           → 30% holes damage
    ('45', 'holes', 0.20),   # 20%           → 45% holes damage
]

def load_mask_cache(folder, suffix_filter=None):
    """Load all matching masks from a folder into a dict."""
    files = [f for f in os.listdir(folder) if f.endswith('.png')]
    if suffix_filter:
        files = [f for f in files if suffix_filter in f]
    cache = {}
    for fname in tqdm(files, desc=f"  Caching {os.path.basename(folder)}", leave=False):
        cache[fname] = load_mask(os.path.join(folder, fname))
    return cache

def create_corrupted_images(split):
    original_dir  = os.path.join(PROCESSED, split, 'original')
    masked_dir    = os.path.join(PROCESSED, split, 'masked')
    mask_save_dir = os.path.join(PROCESSED, split, 'masks')       # applied masks only
    crack_dir     = os.path.join(PROCESSED, split, 'augmented_masks')
    holes_dir     = os.path.join(PROCESSED, split, 'augmented_holes')
    meta_path     = os.path.join(PROCESSED, split, 'metadata.json')

    os.makedirs(masked_dir, exist_ok=True)
    os.makedirs(mask_save_dir, exist_ok=True)

    # Load mask caches (split by coverage suffix)
    print(f"  Loading mask caches for {split}...")
    crack_caches = {
        '05': load_mask_cache(crack_dir, '_cov05'),
        '10': load_mask_cache(crack_dir, '_cov10'),
        '15': load_mask_cache(crack_dir, '_cov15'),
    }
    holes_caches = {
        '30': load_mask_cache(holes_dir, '_cov30') if os.path.exists(holes_dir) else {},
        '45': load_mask_cache(holes_dir, '_cov45') if os.path.exists(holes_dir) else {},
    }

    # Verify no empty caches
    for cov, cache in {**crack_caches, **holes_caches}.items():
        assert len(cache) > 0, (
            f"FAIL: empty mask cache for cov={cov} in {split}. "
            f"Run cells 7 and 8 first."
        )

    print(f"  Mask cache sizes — "
          f"crack: 05={len(crack_caches['05'])} 10={len(crack_caches['10'])} "
          f"15={len(crack_caches['15'])}  "
          f"holes: 30={len(holes_caches['30'])} 45={len(holes_caches['45'])}")

    image_files = sorted([f for f in os.listdir(original_dir) if f.endswith('.png')])

    # Pre-assign coverage levels to achieve exact distribution
    # This is truly stratified, not random.choice per image
    n_images     = len(image_files)
    assignments  = []
    for cov_str, mask_type, fraction in COVERAGE_DIST:
        count = round(n_images * fraction)
        assignments.extend([(cov_str, mask_type)] * count)
    # handle rounding: pad or trim to match exactly
    while len(assignments) < n_images:
        assignments.append(assignments[-1])
    assignments = assignments[:n_images]

    rng = np.random.default_rng(SPLIT_SEEDS[split])
    rng.shuffle(assignments)

    metadata = []

    for img_file, (cov_str, mask_type) in tqdm(
        zip(image_files, assignments),
        desc=f"  Corrupting {split}",
        total=n_images
    ):
        # Load original
        orig_np = np.array(Image.open(
            os.path.join(original_dir, img_file)
        ).convert('RGB')) / 255.0

        # Pick mask from appropriate cache
        cache = crack_caches[cov_str] if mask_type == 'crack' else holes_caches[cov_str]
        mask_fname = rng.choice(list(cache.keys()))
        mask       = cache[mask_fname]

        # Spatial shift
        dx = int(rng.integers(-50, 51))
        dy = int(rng.integers(-50, 51))
        h, w = mask.shape
        M_shift = np.float32([[1, 0, dx], [0, 1, dy]])
        shifted = cv2.warpAffine(
            mask, M_shift, (w, h),
            flags=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=0
        )

        # Apply corruption (zero out masked pixels)
        corrupted            = orig_np.copy()
        mask_bool            = shifted > 0
        corrupted[mask_bool] = 0.0

        # Save corrupted image
        Image.fromarray(
            (corrupted * 255).astype(np.uint8)
        ).save(os.path.join(masked_dir, img_file))

        # Save applied mask
        save_mask(shifted, os.path.join(mask_save_dir, img_file))

        actual_coverage = float(np.sum(mask_bool) / (512 * 512) * 100)
        metadata.append({
            'image_name':      img_file,
            'mask_file':       mask_fname,
            'mask_type':       mask_type,
            'target_coverage': int(cov_str),
            'actual_coverage': round(actual_coverage, 2),
            'dx':              dx,
            'dy':              dy,
        })

    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"  {split}: {n_images} images corrupted, metadata saved.")
    return metadata

all_metadata = {}
for split in ['train', 'val', 'test']:
    all_metadata[split] = create_corrupted_images(split)



  Loading mask caches for train...


  Mask cache sizes — crack: 05=306 10=306 15=306  holes: 30=100 45=100


  Corrupting train: 100%|██████████| 1417/1417 [00:34<00:00, 41.63it/s]


  train: 1417 images corrupted, metadata saved.
  Loading mask caches for val...


  Mask cache sizes — crack: 05=63 10=63 15=63  holes: 30=30 45=30


  Corrupting val: 100%|██████████| 303/303 [00:07<00:00, 40.59it/s]


  val: 303 images corrupted, metadata saved.
  Loading mask caches for test...


  Mask cache sizes — crack: 05=72 10=72 15=72  holes: 30=30 45=30


  Corrupting test: 100%|██████████| 305/305 [00:07<00:00, 41.00it/s]

  test: 305 images corrupted, metadata saved.


In [11]:

def verify_coverage_distribution(split):
    meta_path = os.path.join(PROCESSED, split, 'metadata.json')
    with open(meta_path, 'r') as f:
        metadata = json.load(f)

    total = len(metadata)
    # Count by target coverage
    by_target = {}
    for entry in metadata:
        t = entry['target_coverage']
        by_target[t] = by_target.get(t, 0) + 1

    # Count by actual coverage (±3% tolerance around each target)
    targets = [5, 10, 15, 30, 45]
    by_actual = {t: 0 for t in targets}
    unmatched = 0
    for entry in metadata:
        cov = entry['actual_coverage']
        matched = False
        for t in targets:
            if abs(cov - t) <= 4.0:
                by_actual[t] += 1
                matched = True
                break
        if not matched:
            unmatched += 1

    print(f"\n{split.upper()} Coverage Distribution  ({total} images total)")
    print(f"  {'Target':>8}  {'Assigned':>10}  {'Actual (±4%)':>14}  {'% of total':>12}")
    print("  " + "-" * 50)
    for t in targets:
        assigned = by_target.get(t, 0)
        actual   = by_actual[t]
        print(f"  {t:>6}%  {assigned:>10}  {actual:>14}  {actual/total*100:>11.1f}%")
    if unmatched:
        print(f"  {'Unmatched':>8}  {'':>10}  {unmatched:>14}  {unmatched/total*100:>11.1f}%")
        print(f"  WARNING: {unmatched} images don't match any target bucket. "
              f"Check mask generation.")
    else:
        print(f"  All {total} images matched to a coverage bucket. OK")

    # Count by mask type
    type_counts = {}
    for entry in metadata:
        t = entry.get('mask_type', 'unknown')
        type_counts[t] = type_counts.get(t, 0) + 1
    print(f"  Mask types: {type_counts}")

for split in ['train', 'val', 'test']:
    verify_coverage_distribution(split)

# Sanity check: masks/ should only contain per-image applied masks (same count as original)
print("\nFolder count check:")
for split in ['train', 'val', 'test']:
    orig  = len([f for f in os.listdir(os.path.join(PROCESSED, split, 'original')) if f.endswith('.png')])
    corr  = len([f for f in os.listdir(os.path.join(PROCESSED, split, 'masked'))   if f.endswith('.png')])
    masks = len([f for f in os.listdir(os.path.join(PROCESSED, split, 'masks'))    if f.endswith('.png')])
    pool  = len([f for f in os.listdir(os.path.join(PROCESSED, split, 'mask_pool')) if f.endswith('.png')])
    ok    = "OK" if orig == corr == masks else "MISMATCH"
    print(f"  {split}: original={orig} masked={corr} masks={masks} "
          f"mask_pool={pool}  {ok}")
    assert orig == corr == masks, f"Count mismatch in {split}"
print("All count checks passed. No orphan cleanup needed.")




TRAIN Coverage Distribution  (1417 images total)
    Target    Assigned    Actual (±4%)    % of total
  --------------------------------------------------
       5%         283             482         34.0%
      10%         283             306         21.6%
      15%         283              61          4.3%
      30%         283             280         19.8%
      45%         285             265         18.7%
  Unmatched                          23          1.6%
  Mask types: {'holes': 568, 'crack': 849}

VAL Coverage Distribution  (303 images total)
    Target    Assigned    Actual (±4%)    % of total
  --------------------------------------------------
       5%          61              98         32.3%
      10%          61              79         26.1%
      15%          61               6          2.0%
      30%          61              60         19.8%
      45%          59              59         19.5%
  Unmatched                           1          0.3%
  Mask types: {'crac

In [12]:

SIGMA            = 3.0
FLOW_OFFSET_DEG  = 90.0
COHERENCE_THRESH = 0.05
FAST_DEV_RUN     = False


def compute_orientation_and_coherence(image_gray, sigma=3.0, flow_offset_deg=90.0):
    Ix = cv2.Sobel(image_gray, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(image_gray, cv2.CV_64F, 0, 1, ksize=3)
    Ixx = gaussian_filter(Ix * Ix, sigma)
    Ixy = gaussian_filter(Ix * Iy, sigma)
    Iyy = gaussian_filter(Iy * Iy, sigma)
    theta_deg = (np.degrees(0.5 * np.arctan2(2.0 * Ixy, Ixx - Iyy))
                 + float(flow_offset_deg)) % 180.0
    diff      = Ixx - Iyy
    coherence = np.clip(
        np.sqrt(diff*diff + 4.0*Ixy*Ixy) / (Ixx + Iyy + 1e-8),
        0.0, 1.0
    )
    return theta_deg.astype(np.float32), coherence.astype(np.float32)


def compute_edge_strength_map(image_gray):
    sx  = cv2.Sobel(image_gray, cv2.CV_64F, 1, 0, ksize=3)
    sy  = cv2.Sobel(image_gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sx*sx + sy*sy)
    p99 = np.percentile(mag, 99)
    return np.clip(mag / (p99 + 1e-8), 0.0, 1.0).astype(np.float32)


def compute_spatial_angular_histograms(orientation_map, coherence_map,
                                       grid_size, coherence_thresh=0.05):
    h, w       = orientation_map.shape
    rh, rw     = h // grid_size, w // grid_size
    n_regions  = grid_size * grid_size
    histograms = np.zeros((n_regions, NUM_BINS), dtype=np.float64)
    bin_edges  = np.linspace(0.0, 180.0, NUM_BINS + 1)
    for i in range(grid_size):
        for j in range(grid_size):
            ridx  = i * grid_size + j
            theta = orientation_map[i*rh:(i+1)*rh, j*rw:(j+1)*rw].ravel()
            coh   = coherence_map[i*rh:(i+1)*rh, j*rw:(j+1)*rw].ravel()
            m     = coh >= coherence_thresh
            if not np.any(m):
                histograms[ridx] = 1.0 / NUM_BINS
                continue
            hist, _ = np.histogram(theta[m], bins=bin_edges, weights=coh[m])
            total   = hist.sum()
            histograms[ridx] = hist / total if total > 0 else 1.0 / NUM_BINS
    return histograms.astype(np.float32)


def orientation_float16_safe(orient_f32):
    max_below = np.nextafter(np.float32(180.0), np.float32(0.0))
    o16       = np.minimum(orient_f32, max_below).astype(np.float16)
    o16[o16 == np.float16(180.0)] = np.float16(0.0)
    return o16


def build_brushstroke_h5(proc_dir, h5_path, sigma=3.0, flow_offset_deg=90.0,
                          coherence_thresh=0.05, compression_level=4,
                          fast_dev_run=False):
    split_dirs = {
        s: os.path.join(proc_dir, s, 'original')
        for s in ['train', 'val', 'test']
    }
    print("=" * 70)
    print("H5 EXTRACTION")
    print(f"  sigma={sigma}  flow_offset={flow_offset_deg}°  "
          f"coh_thresh={coherence_thresh}  compression={compression_level}")
    print("=" * 70)

    with h5py.File(h5_path, 'w') as f:
        # FIX: store extraction config as H5 root attributes
        f.attrs['sigma']             = sigma
        f.attrs['flow_offset_deg']   = flow_offset_deg
        f.attrs['coherence_thresh']  = coherence_thresh
        f.attrs['num_bins']          = NUM_BINS
        f.attrs['created']           = datetime.datetime.now().isoformat()
        f.attrs['fast_dev_run']      = int(fast_dev_run)

        for split, split_dir in split_dirs.items():
            if not os.path.exists(split_dir):
                print(f"  Skipping {split}: {split_dir} not found")
                continue

            grp   = f.create_group(split)
            files = sorted([fn for fn in os.listdir(split_dir) if fn.endswith('.png')])
            if fast_dev_run:
                files = files[:5]

            print(f"\n  {split}: {len(files)} images")
            for fn in tqdm(files, desc=f"  {split}"):
                img = np.array(
                    Image.open(os.path.join(split_dir, fn)).convert('L'),
                    dtype=np.float32
                ) / 255.0
                H, W = img.shape

                orient, coh = compute_orientation_and_coherence(
                    img, sigma=sigma, flow_offset_deg=flow_offset_deg
                )
                edge = compute_edge_strength_map(img)
                h4   = compute_spatial_angular_histograms(
                    orient, coh, grid_size=4, coherence_thresh=coherence_thresh
                )
                h8   = compute_spatial_angular_histograms(
                    orient, coh, grid_size=8, coherence_thresh=coherence_thresh
                )

                ig = grp.create_group(fn)

                orient16 = orientation_float16_safe(orient).reshape(H, W, 1)
                coh16    = coh.astype(np.float16).reshape(H, W, 1)
                edge16   = edge.astype(np.float16).reshape(H, W, 1)

                # FIX: chunk size (64, 64, 1) — much better gzip ratio than (512,512,1)
                chunk = (64, 64, 1)
                opts  = dict(compression='gzip', compression_opts=int(compression_level))

                ig.create_dataset('orientation_field',  data=orient16, chunks=chunk, **opts)
                ig.create_dataset('coherence_map',      data=coh16,    chunks=chunk, **opts)
                ig.create_dataset('edge_strength_map',  data=edge16,   chunks=chunk, **opts)
                ig.create_dataset('spatial_hist_4x4',   data=h4.astype(np.float32))
                ig.create_dataset('spatial_hist_8x8',   data=h8.astype(np.float32))

    size_mb = os.path.getsize(h5_path) / 1e6
    print(f"\nH5 saved: {h5_path}  ({size_mb:.1f} MB)")
    print(f"Expected ~900–1200 MB with (64,64,1) chunks. "
          f"If still >2 GB, lower compression or check image count.")


build_brushstroke_h5(
    PROCESSED, H5_PATH,
    sigma=SIGMA,
    flow_offset_deg=FLOW_OFFSET_DEG,
    coherence_thresh=COHERENCE_THRESH,
    compression_level=4,
    fast_dev_run=FAST_DEV_RUN
)


H5 EXTRACTION
  sigma=3.0  flow_offset=90.0°  coh_thresh=0.05  compression=4

  train: 1417 images


  train: 100%|██████████| 1417/1417 [02:24<00:00,  9.80it/s]



  val: 303 images


  val: 100%|██████████| 303/303 [00:26<00:00, 11.30it/s]



  test: 305 images


  test: 100%|██████████| 305/305 [00:27<00:00, 11.15it/s]



H5 saved: /Users/althafali/Downloads/ARTIFEX/implementation/brushstroke_features.h5  (2637.5 MB)
Expected ~900–1200 MB with (64,64,1) chunks. If still >2 GB, lower compression or check image count.


In [13]:

def verify_full_pipeline(proc_dir, h5_path, n_sample=10):
    print("=" * 70)
    print("FULL PIPELINE VERIFICATION")
    print("=" * 70)
    all_ok = True

    # 1. Check H5 config attributes
    with h5py.File(h5_path, 'r') as f:
        attrs = dict(f.attrs)
        print(f"\nH5 config: {attrs}")
        assert 'sigma' in attrs,            "FAIL: sigma attr missing from H5"
        assert 'flow_offset_deg' in attrs,  "FAIL: flow_offset_deg attr missing"
        print("H5 config attributes: OK")

    # 2. Per-split checks
    with h5py.File(h5_path, 'r') as f:
        for split in ['train', 'val', 'test']:
            print(f"\n  {split.upper()}")
            orig_dir = os.path.join(proc_dir, split, 'original')
            corr_dir = os.path.join(proc_dir, split, 'masked')
            mask_dir = os.path.join(proc_dir, split, 'masks')
            pool_dir = os.path.join(proc_dir, split, 'mask_pool')

            orig_files  = set(fn for fn in os.listdir(orig_dir) if fn.endswith('.png'))
            corr_files  = set(fn for fn in os.listdir(corr_dir) if fn.endswith('.png'))
            mask_files  = set(fn for fn in os.listdir(mask_dir) if fn.endswith('.png'))
            pool_files  = set(fn for fn in os.listdir(pool_dir) if fn.endswith('.png'))

            # Count check
            if orig_files == corr_files == mask_files:
                print(f"    File counts: {len(orig_files)} orig/masked/masks  OK")
            else:
                print(f"    FAIL: count mismatch orig={len(orig_files)} "
                      f"corr={len(corr_files)} mask={len(mask_files)}")
                all_ok = False

            # mask_pool should not contain processed filenames
            overlap = pool_files & orig_files
            if overlap:
                print(f"    FAIL: {len(overlap)} processed filenames found in mask_pool — "
                      f"folder contamination")
                all_ok = False
            else:
                print(f"    mask_pool isolated ({len(pool_files)} base masks, no overlap): OK")

            # H5 alignment
            if split not in f:
                print(f"    FAIL: {split} missing from H5")
                all_ok = False
                continue

            h5_names = set(f[split].keys())
            missing  = orig_files - h5_names
            extra    = h5_names - orig_files
            if missing:
                print(f"    FAIL: {len(missing)} images missing from H5")
                all_ok = False
            else:
                print(f"    H5 alignment: all {len(orig_files)} images present  OK")
            if extra:
                print(f"    WARN: {len(extra)} extra H5 entries")

            # Sample validation
            sample_names = sorted(h5_names)[:n_sample]
            orient_fails = 0
            nan_fails    = 0
            hist_fails   = 0
            for fn in sample_names:
                ig  = f[split][fn]
                o   = ig['orientation_field'][:].astype(np.float32)
                c   = ig['coherence_map'][:].astype(np.float32)
                e   = ig['edge_strength_map'][:].astype(np.float32)
                h4  = ig['spatial_hist_4x4'][:]
                h8  = ig['spatial_hist_8x8'][:]
                if not (o.min() >= 0 and o.max() < 180.0):
                    orient_fails += 1
                if not (np.isfinite(o).all() and np.isfinite(c).all()
                        and np.isfinite(e).all()):
                    nan_fails += 1
                if not (np.allclose(h4.sum(axis=1), 1.0, atol=1e-4)
                        and np.allclose(h8.sum(axis=1), 1.0, atol=1e-4)):
                    hist_fails += 1

            if orient_fails or nan_fails or hist_fails:
                print(f"    FAIL in {n_sample} samples: orient={orient_fails} "
                      f"nan={nan_fails} hist={hist_fails}")
                all_ok = False
            else:
                print(f"    {n_sample}-sample checks (orient/nan/hist): all OK")

    size_mb = os.path.getsize(h5_path) / 1e6
    print(f"\nH5 file size: {size_mb:.1f} MB")
    print("\n" + "=" * 70)
    print("RESULT:", "ALL CHECKS PASSED" if all_ok else "FAILURES FOUND — see above")
    print("=" * 70)
    return all_ok

verify_full_pipeline(PROCESSED, H5_PATH)

FULL PIPELINE VERIFICATION

H5 config: {'coherence_thresh': np.float64(0.05), 'created': '2026-02-24T14:00:23.989232', 'fast_dev_run': np.int64(0), 'flow_offset_deg': np.float64(90.0), 'num_bins': np.int64(8), 'sigma': np.float64(3.0)}
H5 config attributes: OK

  TRAIN
    File counts: 1417 orig/masked/masks  OK
    mask_pool isolated (34 base masks, no overlap): OK
    H5 alignment: all 1417 images present  OK
    10-sample checks (orient/nan/hist): all OK

  VAL
    File counts: 303 orig/masked/masks  OK
    mask_pool isolated (7 base masks, no overlap): OK
    H5 alignment: all 303 images present  OK
    10-sample checks (orient/nan/hist): all OK

  TEST
    File counts: 305 orig/masked/masks  OK
    mask_pool isolated (8 base masks, no overlap): OK
    H5 alignment: all 305 images present  OK
    10-sample checks (orient/nan/hist): all OK

H5 file size: 2637.5 MB

RESULT: ALL CHECKS PASSED


True